# 🚀 YOLO Disease Detection with Data Augmentation

## Complete Step-by-Step Guide for Google Colab

This notebook will:
1. ✅ Download your dataset from Google Drive (link: 1wqsh54_CFKgyxHQefnOdHMPdjRyaSPJg)
2. ✅ Apply augmentation to create 6000-8000 images
3. ✅ Train YOLOv8 for disease detection
4. ✅ Run inference and visualize results

### ⏱️ Expected Runtime
- **With GPU (Recommended):** 2-3 hours total
- **CPU Only:** 6-8 hours (not recommended)

### 📋 Pre-Colab Checklist
- [ ] You have Google Drive access
- [ ] Enable GPU before running (Runtime → Change runtime type → GPU)
- [ ] Folder link is accessible in your Google Drive
- [ ] At least 10GB free space in Google Drive

---

## 🎯 How to Run This Notebook in Google Colab

### STEP 1: Open in Colab
1. Copy this notebook URL or upload this file
2. Open in [colab.research.google.com](https://colab.research.google.com)

### STEP 2: Enable GPU (CRITICAL!)
1. Click **Runtime** menu
2. Select **Change runtime type**
3. Choose **Hardware accelerator: GPU** (T4 or V100)
4. Click **Save**

### STEP 3: Run Cells in Order
Execute each cell from top to bottom. Read the output carefully.

### STEP 4: Save Results
- Trained model saved to Google Drive automatically
- Download results from Drive when complete

---

# CELL 1️⃣ - Mount Google Drive & Install Dependencies
**⏱️ Runtime: 2-3 minutes**

This cell will:
- Mount your Google Drive
- Install YOLOv8 and required libraries
- Verify everything is ready

In [ ]:
# CELL 1: Mount Google Drive & Install Dependencies
# ====================================================

# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted successfully!")

# Step 2: Install required libraries
import subprocess
import sys

print("\n📦 Installing YOLOv8 and dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "albumentations"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "opencv-python"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])

print("✅ All dependencies installed!")

# Step 3: Verify installations
import cv2
import numpy as np
from ultralytics import YOLO
import albumentations as A
import yaml
import os
import shutil

print("\n✅ All imports successful!")
print(f"OpenCV version: {cv2.__version__}")
print("YOLOv8: Ready ✓")
print("Albumentations: Ready ✓")

---

# CELL 2️⃣ - Load and Explore Dataset
**⏱️ Runtime: 1-2 minutes**

This cell will:
- Access the Google Drive folder with your processed data
- Load and display sample images
- Check dataset statistics
- Show disease distribution

In [ ]:
# CELL 2: Load and Explore Dataset
# ==================================

import matplotlib.pyplot as plt
from collections import defaultdict
import random

# Configuration - MODIFY THESE IF NEEDED
DRIVE_FOLDER_ID = "1wqsh54_CFKgyxHQefnOdHMPdjRyaSPJg"  # Your folder ID
DATASET_PATH = "/content/drive/MyDrive/Banana_Leaf_Processed"  # Adjust if needed

# Check if folder exists, if not use alternative path
if not os.path.exists(DATASET_PATH):
    print(f"⚠️  Path not found: {DATASET_PATH}")
    print("\n🔍 Searching for dataset folders in Google Drive...")
    
    # List contents of My Drive
    drive_path = "/content/drive/MyDrive"
    if os.path.exists(drive_path):
        folders = os.listdir(drive_path)
        print(f"\nFolders in My Drive:\n{folders}")
    
    # Try alternative names
    alternatives = [
        "/content/drive/MyDrive/processed",
        "/content/drive/MyDrive/Banana",
        "/content/drive/MyDrive/banana_leaf_processed",
        "/content/drive/MyDrive/split"
    ]
    
    for alt in alternatives:
        if os.path.exists(alt):
            DATASET_PATH = alt
            print(f"\n✅ Found dataset at: {DATASET_PATH}")
            break

print(f"📁 Using dataset path: {DATASET_PATH}\n")

# Find all image files
image_extensions = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')
all_images = []
class_distribution = defaultdict(int)

for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.lower().endswith(image_extensions):
            full_path = os.path.join(root, file)
            all_images.append(full_path)
            # Extract class from folder structure
            class_name = os.path.basename(root)
            class_distribution[class_name] += 1

print(f"📊 DATASET STATISTICS")
print(f"=" * 50)
print(f"Total images found: {len(all_images)}")
print(f"\nDisease Classes:")
for class_name, count in sorted(class_distribution.items()):
    print(f"  • {class_name}: {count} images")

# Display sample images
if len(all_images) > 0:
    print(f"\n🖼️ Displaying 6 random sample images...")
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    sample_indices = random.sample(range(len(all_images)), min(6, len(all_images)))
    
    for idx, ax in enumerate(axes):
        if idx < len(sample_indices):
            img_path = all_images[sample_indices[idx]]
            img = cv2.imread(img_path)
            if img is not None:
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                ax.imshow(img_rgb)
                class_name = os.path.basename(os.path.dirname(img_path))
                ax.set_title(f"Class: {class_name}\nSize: {img_rgb.shape}", fontsize=10)
            ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('/content/sample_images.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("✅ Sample images displayed!")
else:
    print("⚠️ No images found. Check the DATASET_PATH variable.")

---

# CELL 3️⃣ - Data Augmentation Pipeline
**⏱️ Runtime: 30-45 minutes** ⏳

This cell will:
- Create 5 different augmentation strategies
- Generate 6000-8000 images from your original dataset
- Save augmented images to organized folders
- Display augmentation statistics

### Augmentation Techniques Used:
1. **Rotation** (±45°)
2. **Zoom** (0.8-1.3x)
3. **Flip** (horizontal & vertical)
4. **Brightness/Contrast** adjustments
5. **Blur & Noise** for robustness
6. **Perspective** transforms
7. **Elastic** distortions

In [ ]:
# CELL 3: Data Augmentation Pipeline
# ===================================

# Configuration for augmentation
AUGMENTED_OUTPUT_PATH = "/content/drive/MyDrive/YOLO_Augmented_Dataset"
TARGET_IMAGES_PER_CLASS = 1500  # Will generate ~1500 images per class (6000 total for 4 classes)
IMAGES_PER_ORIGINAL = 5  # Each original image will be augmented 5 times

# Create output directory
os.makedirs(AUGMENTED_OUTPUT_PATH, exist_ok=True)

print("🎨 DATA AUGMENTATION PIPELINE")
print("=" * 60)
print(f"Target: {TARGET_IMAGES_PER_CLASS * len(class_distribution)} total images")
print(f"Target per class: {TARGET_IMAGES_PER_CLASS} images")
print(f"Output path: {AUGMENTED_OUTPUT_PATH}\n")

# Define 5 augmentation strategies
augmentation_pipelines = {
    "aggressive": A.Compose([
        A.Rotate(limit=50, p=0.9),
        A.Perspective(scale=(0.05, 0.1), p=0.7),
        A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.5),
        A.GaussNoise(p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
        A.Blur(blur_limit=5, p=0.3),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'])),
    
    "moderate": A.Compose([
        A.Rotate(limit=30, p=0.8),
        A.Perspective(scale=(0.03, 0.07), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.6),
        A.Blur(blur_limit=3, p=0.3),
        A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.4),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'])),
    
    "light": A.Compose([
        A.Rotate(limit=20, p=0.7),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=20, val_shift_limit=15, p=0.3),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'])),
    
    "color_jitter": A.Compose([
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.8),
        A.HueSaturationValue(hue_shift_limit=30, sat_shift_limit=40, val_shift_limit=30, p=0.8),
        A.GaussNoise(p=0.2),
        A.Blur(blur_limit=3, p=0.2),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'])),
    
    "geometric": A.Compose([
        A.Rotate(limit=40, p=0.8),
        A.Transpose(p=0.5),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Perspective(scale=(0.05, 0.1), p=0.6),
        A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.4),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'])),
}

augmentation_count = defaultdict(int)
total_augmented = 0

# Process each class
for class_name in class_distribution.keys():
    class_output_dir = os.path.join(AUGMENTED_OUTPUT_PATH, class_name)
    os.makedirs(class_output_dir, exist_ok=True)
    
    # Get all images for this class
    class_images = [img for img in all_images if os.path.basename(os.path.dirname(img)) == class_name]
    
    print(f"\n🔄 Augmenting class: {class_name}")
    print(f"   Original images: {len(class_images)}")
    
    # Calculate how many times to augment each image
    if len(class_images) > 0:
        augmentations_needed = max(1, TARGET_IMAGES_PER_CLASS // len(class_images))
    else:
        continue
    
    augmented_count = 0
    
    # Augment each image
    for img_idx, img_path in enumerate(class_images):
        img = cv2.imread(img_path)
        if img is None:
            continue
        
        # Save original
        original_name = f"{class_name}_{img_idx}_original.jpg"
        cv2.imwrite(os.path.join(class_output_dir, original_name), img)
        augmented_count += 1
        
        # Apply each augmentation strategy
        for aug_idx, (aug_name, transform) in enumerate(augmentation_pipelines.items()):
            if augmented_count >= TARGET_IMAGES_PER_CLASS:
                break
            
            for aug_iteration in range(IMAGES_PER_ORIGINAL):
                if augmented_count >= TARGET_IMAGES_PER_CLASS:
                    break
                
                try:
                    # Apply augmentation (without bboxes for simpler implementation)
                    augmented = transform(image=img, bboxes=[], labels=[])
                    augmented_img = augmented['image']
                    
                    # Save augmented image
                    aug_filename = f"{class_name}_{img_idx}_{aug_name}_{aug_iteration}.jpg"
                    cv2.imwrite(os.path.join(class_output_dir, aug_filename), augmented_img)
                    augmented_count += 1
                    
                except Exception as e:
                    print(f"   ⚠️ Error augmenting {img_path}: {str(e)[:50]}")
                    continue
        
        if (img_idx + 1) % max(1, len(class_images) // 5) == 0:
            print(f"   ✓ Processed {img_idx + 1}/{len(class_images)} images ({augmented_count} augmented)")
    
    print(f"   ✅ Completed: {augmented_count} images for {class_name}")
    augmentation_count[class_name] = augmented_count
    total_augmented += augmented_count

# Summary
print(f"\n" + "=" * 60)
print(f"✅ AUGMENTATION COMPLETE!")
print(f"=" * 60)
print(f"Total augmented images: {total_augmented}")
for class_name, count in sorted(augmentation_count.items()):
    print(f"  • {class_name}: {count} images")
print(f"\n📁 Saved to: {AUGMENTED_OUTPUT_PATH}")

---

# CELL 4️⃣ - Prepare Data for YOLO
**⏱️ Runtime: 5 minutes**

This cell will:
- Create YOLO-compatible dataset structure
- Split data into train/val/test (80/10/10)
- Generate dataset.yaml configuration file
- Verify YOLO format compliance

### YOLO Dataset Structure:
```
dataset/
├── images/
│   ├── train/
│   ├── val/
│   └── test/
├── labels/
│   ├── train/
│   ├── val/
│   └── test/
└── dataset.yaml
```

In [ ]:
# CELL 4: Prepare Data for YOLO
# ==============================

# YOLO dataset path
YOLO_DATASET_PATH = "/content/yolo_dataset"

# Create YOLO directory structure
splits = ['train', 'val', 'test']
for split in splits:
    os.makedirs(os.path.join(YOLO_DATASET_PATH, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DATASET_PATH, 'labels', split), exist_ok=True)

print("📦 PREPARING YOLO DATASET")
print("=" * 60)

# Split configuration
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.1
TEST_SPLIT = 0.1

class_to_id = {}
split_counts = defaultdict(lambda: defaultdict(int))
all_augmented_images = []

# Get all augmented images
for class_name in os.listdir(AUGMENTED_OUTPUT_PATH):
    class_path = os.path.join(AUGMENTED_OUTPUT_PATH, class_name)
    if os.path.isdir(class_path):
        class_to_id[class_name] = len(class_to_id)
        for img_file in os.listdir(class_path):
            if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_augmented_images.append((os.path.join(class_path, img_file), class_name))

print(f"Total augmented images: {len(all_augmented_images)}")
print(f"Classes: {class_to_id}")

# Shuffle and split
random.shuffle(all_augmented_images)
train_count = int(len(all_augmented_images) * TRAIN_SPLIT)
val_count = int(len(all_augmented_images) * VAL_SPLIT)

train_images = all_augmented_images[:train_count]
val_images = all_augmented_images[train_count:train_count + val_count]
test_images = all_augmented_images[train_count + val_count:]

splits_data = {
    'train': train_images,
    'val': val_images,
    'test': test_images
}

# Copy images and create labels
print(f"\n🔄 Organizing data into YOLO format...")
for split_name, images in splits_data.items():
    for img_path, class_name in images:
        # Copy image
        img_filename = os.path.basename(img_path)
        dest_img_path = os.path.join(YOLO_DATASET_PATH, 'images', split_name, img_filename)
        shutil.copy2(img_path, dest_img_path)
        
        # Create corresponding label file
        label_filename = img_filename.rsplit('.', 1)[0] + '.txt'
        label_path = os.path.join(YOLO_DATASET_PATH, 'labels', split_name, label_filename)
        
        # For classification: just write the class ID
        # For detection: we would write bounding box coordinates
        # Using classification format for simplicity
        class_id = class_to_id[class_name]
        with open(label_path, 'w') as f:
            f.write(str(class_id))
        
        split_counts[split_name][class_name] += 1
    
    print(f"  ✓ {split_name}: {len(images)} images")

# Create dataset.yaml
dataset_yaml = {
    'path': YOLO_DATASET_PATH,
    'train': os.path.join(YOLO_DATASET_PATH, 'images', 'train'),
    'val': os.path.join(YOLO_DATASET_PATH, 'images', 'val'),
    'test': os.path.join(YOLO_DATASET_PATH, 'images', 'test'),
    'nc': len(class_to_id),
    'names': {v: k for k, v in class_to_id.items()}
}

yaml_path = os.path.join(YOLO_DATASET_PATH, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print(f"\n✅ dataset.yaml created at: {yaml_path}")
print(f"\n📊 SPLIT STATISTICS:")
for split, counts in sorted(split_counts.items()):
    total = sum(counts.values())
    print(f"\n{split.upper()}: {total} images")
    for class_name, count in sorted(counts.items()):
        percentage = (count / total) * 100
        print(f"  • {class_name}: {count} ({percentage:.1f}%)")

print(f"\n✅ YOLO dataset ready at: {YOLO_DATASET_PATH}")

---

# CELL 5️⃣ - Train YOLO Model
**⏱️ Runtime: 45-90 minutes** ⏳

This cell will:
- Load pretrained YOLOv8 model
- Configure training parameters
- Train model on augmented dataset
- Track training metrics
- Save trained model

### Model Options:
- **YOLOv8n** (nano) - Fast, low memory - Recommended for Colab
- **YOLOv8s** (small) - Balanced
- **YOLOv8m** (medium) - Better accuracy
- **YOLOv8l** (large) - Best accuracy, more memory

In [ ]:
# CELL 5: Train YOLO Model
# =========================

print("🚀 TRAINING YOLO MODEL")
print("=" * 60)

# Load YOLOv8 model
# Options: yolov8n, yolov8s, yolov8m, yolov8l
MODEL_SIZE = "n"  # nano (fastest, recommended for Colab)
print(f"📦 Loading YOLOv8{MODEL_SIZE} model...")

try:
    model = YOLO(f'yolov8{MODEL_SIZE}-cls.pt')  # cls for classification
    print(f"✅ YOLOv8{MODEL_SIZE} model loaded successfully!")
except Exception as e:
    print(f"⚠️ Error loading model: {e}")
    print("Trying alternative download...")
    model = YOLO(f'yolov8{MODEL_SIZE}-cls.pt')

# Training parameters
EPOCHS = 50
BATCH_SIZE = 16  # Reduce if out of memory
PATIENCE = 10  # Early stopping patience
LEARNING_RATE = 0.001
IMG_SIZE = 640

# Create output directory for results
RESULTS_DIR = "/content/drive/MyDrive/YOLO_Results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"\n⚙️ TRAINING CONFIGURATION:")
print(f"  • Model: YOLOv8{MODEL_SIZE}")
print(f"  • Epochs: {EPOCHS}")
print(f"  • Batch Size: {BATCH_SIZE}")
print(f"  • Learning Rate: {LEARNING_RATE}")
print(f"  • Image Size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  • Dataset: {YOLO_DATASET_PATH}")
print(f"  • Output: {RESULTS_DIR}")

# Train the model
print(f"\n🎯 Starting training...\n")
try:
    results = model.train(
        data=yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        patience=PATIENCE,
        device=0,  # 0 for GPU (cuda:0)
        project=RESULTS_DIR,
        name='banana_disease_detector',
        exist_ok=True,
        verbose=True,
        save=True,
        save_txt=True,
        save_conf=True,
        plots=True,
    )
    print(f"\n✅ Training completed successfully!")
    
except Exception as e:
    print(f"❌ Error during training: {e}")
    print("Check GPU availability and memory usage")

# Save model to Google Drive
FINAL_MODEL_PATH = os.path.join(RESULTS_DIR, 'best_model.pt')
try:
    # Find the best model from training
    best_model_path = os.path.join(RESULTS_DIR, 'banana_disease_detector', 'weights', 'best.pt')
    if os.path.exists(best_model_path):
        shutil.copy2(best_model_path, FINAL_MODEL_PATH)
        print(f"✅ Best model saved to: {FINAL_MODEL_PATH}")
    else:
        print(f"⚠️ Best model not found at: {best_model_path}")
except Exception as e:
    print(f"⚠️ Error saving model: {e}")

print(f"\n📊 Training Results:")
print(f"  • Results directory: {RESULTS_DIR}")
print(f"  • Model saved: {FINAL_MODEL_PATH}")

---

# CELL 6️⃣ - Run Disease Detection (Inference)
**⏱️ Runtime: 5-10 minutes**

This cell will:
- Load the trained model
- Run inference on test images
- Display predictions with disease labels
- Show confidence scores
- Visualize results with bounding boxes

In [ ]:
# CELL 6: Run Disease Detection (Inference)
# ==========================================

print("🔍 DISEASE DETECTION - INFERENCE")
print("=" * 60)

# Load the trained model
print("📦 Loading trained model...")
try:
    best_model_path = os.path.join(RESULTS_DIR, 'banana_disease_detector', 'weights', 'best.pt')
    if os.path.exists(best_model_path):
        model = YOLO(best_model_path)
        print(f"✅ Model loaded from: {best_model_path}")
    else:
        print(f"⚠️ Model not found at {best_model_path}")
        print("Using last training model...")
except Exception as e:
    print(f"❌ Error loading model: {e}")

# Get test images
test_images_dir = os.path.join(YOLO_DATASET_PATH, 'images', 'test')
test_image_files = [f for f in os.listdir(test_images_dir) 
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"\n📊 Running inference on {len(test_image_files)} test images...")

# Run predictions on sample test images
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

predictions_summary = defaultdict(lambda: {'total': 0, 'correct': 0})
all_predictions = []

for idx, (ax, img_file) in enumerate(zip(axes, test_image_files[:6])):
    img_path = os.path.join(test_images_dir, img_file)
    img = cv2.imread(img_path)
    
    if img is None:
        continue
    
    # Run YOLO inference
    results = model.predict(img_path, verbose=False, conf=0.25)
    
    # Get prediction
    if results and len(results) > 0:
        result = results[0]
        pred_class_id = int(result.probs.top1)
        pred_confidence = float(result.probs.top1conf)
        pred_class_name = result.names[pred_class_id]
        
        # Get actual class from label file
        label_file = img_file.rsplit('.', 1)[0] + '.txt'
        label_path = os.path.join(YOLO_DATASET_PATH, 'labels', 'test', label_file)
        try:
            with open(label_path, 'r') as f:
                actual_class_id = int(f.read().strip())
                actual_class_name = list(class_to_id.keys())[list(class_to_id.values()).index(actual_class_id)]
        except:
            actual_class_name = "Unknown"
        
        # Store prediction
        all_predictions.append({
            'image': img_file,
            'predicted': pred_class_name,
            'actual': actual_class_name,
            'confidence': pred_confidence,
            'correct': pred_class_name == actual_class_name
        })
        
        # Display image with prediction
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        
        # Color based on correctness
        color = 'green' if pred_class_name == actual_class_name else 'red'
        title = f"Disease: {pred_class_name}\\nConfidence: {pred_confidence:.2%}\\nActual: {actual_class_name}\"\n\"\"\""
        ax.set_title(title, fontsize=10, color=color, fontweight='bold')
        ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'inference_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Inference visualization saved!")

# Print summary
if all_predictions:
    print(f"\n📊 INFERENCE SUMMARY:")
    total = len(all_predictions)
    correct = sum(1 for p in all_predictions if p['correct'])
    accuracy = (correct / total) * 100 if total > 0 else 0
    
    print(f"\nAccuracy on sample: {accuracy:.1f}% ({correct}/{total})")
    print(f"\nSample Predictions:")
    for pred in all_predictions[:5]:
        status = "✅" if pred['correct'] else "❌"
        print(f"  {status} {pred['image'][:30]:30} | Pred: {pred['predicted']:15} | Conf: {pred['confidence']:.2%} | Actual: {pred['actual']}")

---

# CELL 7️⃣ - Evaluate Model Results
**⏱️ Runtime: 3-5 minutes**

This cell will:
- Calculate precision, recall, F1-score
- Show confusion matrix
- Display per-class metrics
- Generate performance report
- Save results to Google Drive

In [ ]:
# CELL 7: Evaluate Model Results
# ===============================

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns

print("📊 MODEL EVALUATION")
print("=" * 60)

# Evaluate on full test set
y_true = []
y_pred = []

test_images_dir = os.path.join(YOLO_DATASET_PATH, 'images', 'test')
test_image_files = [f for f in os.listdir(test_images_dir) 
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"🔄 Evaluating on {len(test_image_files)} test images...")

for img_file in test_image_files:
    img_path = os.path.join(test_images_dir, img_file)
    
    # Get prediction
    results = model.predict(img_path, verbose=False, conf=0.25)
    if results and len(results) > 0:
        result = results[0]
        pred_class_id = int(result.probs.top1)
        pred_class_name = result.names[pred_class_id]
    else:
        pred_class_name = "Unknown"
    
    # Get actual label
    label_file = img_file.rsplit('.', 1)[0] + '.txt'
    label_path = os.path.join(YOLO_DATASET_PATH, 'labels', 'test', label_file)
    try:
        with open(label_path, 'r') as f:
            actual_class_id = int(f.read().strip())
            actual_class_name = list(class_to_id.keys())[list(class_to_id.values()).index(actual_class_id)]
    except:
        actual_class_name = "Unknown"
    
    y_true.append(actual_class_name)
    y_pred.append(pred_class_name)

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred)

print(f"\n✅ EVALUATION RESULTS:")
print(f"=" * 60)
print(f"\nOverall Accuracy: {accuracy:.2%}")

# Per-class metrics
print(f"\n📋 Classification Report:")
print(classification_report(y_true, y_pred, labels=sorted(list(class_to_id.keys()))))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=sorted(list(class_to_id.keys())))

fig, ax = plt.subplots(figsize=(10, 8))
class_names = sorted(list(class_to_id.keys()))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title('Confusion Matrix - YOLO Disease Detection', fontsize=14, fontweight='bold')
ax.set_ylabel('Actual Disease', fontsize=12)
ax.set_xlabel('Predicted Disease', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Confusion matrix saved!")

# Save evaluation report
report_path = os.path.join(RESULTS_DIR, 'evaluation_report.txt')
with open(report_path, 'w') as f:
    f.write("YOLO DISEASE DETECTION - EVALUATION REPORT\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Dataset: Banana Leaf Disease Classification\n")
    f.write(f"Model: YOLOv8{MODEL_SIZE}\n")
    f.write(f"Training Epochs: {EPOCHS}\n")
    f.write(f"Batch Size: {BATCH_SIZE}\n")
    f.write(f"Total Augmented Images: {total_augmented}\n\n")
    f.write(f"Test Set Size: {len(test_image_files)} images\n")
    f.write(f"Overall Accuracy: {accuracy:.2%}\n\n")
    f.write("Classification Report:\n")
    f.write(classification_report(y_true, y_pred, labels=sorted(list(class_to_id.keys()))))

print(f"\n📄 Evaluation report saved to: {report_path}")
print(f"\n✅ EVALUATION COMPLETE!")

---

# CELL 8️⃣ - Final Summary & Download Results
**⏱️ Runtime: 1 minute**

This cell will:
- Generate final summary report
- List all saved files
- Provide instructions to download results
- Show how to use the trained model locally

In [ ]:
# CELL 8: Final Summary & Download Results
# ========================================

print("\n" + "=" * 70)
print(" " * 15 + "🎉 YOLO DISEASE DETECTION PIPELINE COMPLETE! 🎉")
print("=" * 70)

print(f"""
✅ PIPELINE SUMMARY:

📊 DATASET TRANSFORMATION:
  • Original images: {len(class_distribution)} per class (approx)
  • Augmented images: {total_augmented} total
  • Training images: {len(train_images)}
  • Validation images: {len(val_images)}
  • Test images: {len(test_images)}

🏆 MODEL PERFORMANCE:
  • Overall Accuracy: {accuracy:.2%}
  • Model: YOLOv8{MODEL_SIZE}
  • Training epochs: {EPOCHS}
  • Batch size: {BATCH_SIZE}

📁 OUTPUT FILES SAVED TO GOOGLE DRIVE:
  • Augmented dataset: {AUGMENTED_OUTPUT_PATH}
  • YOLO dataset: {YOLO_DATASET_PATH}
  • Results & models: {RESULTS_DIR}
  
  Specific files:
  ✓ best_model.pt - Trained YOLO model
  ✓ confusion_matrix.png - Performance visualization
  ✓ evaluation_report.txt - Detailed metrics
  ✓ inference_results.png - Sample predictions
  ✓ Training plots and losses (in banana_disease_detector folder)

""")

print("\n📋 FILES READY FOR DOWNLOAD FROM GOOGLE DRIVE:")
print("=" * 70)

# List key output files
key_files = [
    ('best_model.pt', 'Trained YOLO model for inference'),
    ('confusion_matrix.png', 'Model performance visualization'),
    ('evaluation_report.txt', 'Detailed accuracy and metrics'),
    ('inference_results.png', 'Sample disease detection results'),
]

print("\nKey Results:")
for file_name, description in key_files:
    file_path = os.path.join(RESULTS_DIR, file_name)
    if os.path.exists(file_path):
        file_size = os.path.getsize(file_path) / (1024*1024)  # MB
        print(f"  ✅ {file_name:30} ({file_size:6.1f} MB) - {description}")
    else:
        print(f"  ⏳ {file_name:30} - {description}")

print(f"\n\n💾 HOW TO USE THE TRAINED MODEL:")
print("=" * 70)
print(f"""
1. Download best_model.pt from Google Drive

2. In Python, load and use the model:
   
   from ultralytics import YOLO
   
   # Load model
   model = YOLO('best_model.pt')
   
   # Predict on image
   results = model.predict('path/to/leaf_image.jpg')
   
   # Get prediction
   result = results[0]
   disease = result.names[int(result.probs.top1)]
   confidence = float(result.probs.top1conf)
   
   print(f"Disease: {{disease}}, Confidence: {{confidence:.2%}}")

3. Or use in Colab again:
   - Upload this notebook
   - Run only CELL 1 (to mount Drive and install)
   - Run CELL 8 (this cell) to see saved models
   - Create a new cell with prediction code above

""")

print(f"\n\n🎯 NEXT STEPS:")
print("=" * 70)
print(f"""
1. ✅ Check results in Google Drive: {RESULTS_DIR}
2. ✅ Download the model (best_model.pt) for local use
3. ✅ Review confusion matrix to understand model performance
4. ✅ Test on new banana leaf images
5. ✅ Consider running with more epochs (EPOCHS=100) for better accuracy

⚠️ IMPORTANT NOTES:
  • Training data is saved in Google Drive automatically
  • Model accuracy depends on dataset quality and size
  • Run with GPU enabled for faster training
  • Can re-run training with different parameters

""")

print("\n" + "=" * 70)
print("✅ ALL COMPLETE! Check Google Drive for your results.")
print("=" * 70)

---

# 📚 COMPLETE COLAB STEP-BY-STEP GUIDE

## 🎯 HOW TO RUN THIS NOTEBOOK IN GOOGLE COLAB (LINE-BY-LINE)

### BEFORE YOU START:
- ✅ Have Google Drive access
- ✅ Make sure your dataset folder link is working
- ✅ Have at least 10GB free storage in Google Drive

---

## STEP 1: OPEN THE NOTEBOOK IN COLAB
1. Go to https://colab.research.google.com
2. Click **File** → **Upload notebook**
3. Choose this notebook (yolo_disease_detection_colab.ipynb)
4. Wait for it to load (usually 10-15 seconds)

---

## STEP 2: ENABLE GPU (CRITICAL!)
1. Click **Runtime** menu (top menu bar)
2. Select **Change runtime type**
3. In the dialog box:
   - Select **GPU** from "Hardware accelerator" dropdown
   - Choose **T4 GPU** (free tier) or **V100** (if available)
4. Click **Save**
5. The runtime will restart

---

## STEP 3: RUN CELLS IN SEQUENTIAL ORDER

### 📍 CELL 1: Mount Google Drive & Install Dependencies
```
What it does: 
- Mounts your Google Drive
- Installs YOLOv8, OpenCV, Albumentations

How to run:
1. Click on the cell
2. Click the ▶️ (Play) button on the left
3. If prompted: Click "Connect to Google Drive"
4. In the popup: Select your Google Account
5. Grant permission
6. Wait for "✅ All imports successful!" message

Expected output:
✅ Google Drive mounted successfully!
✅ All dependencies installed!
✅ All imports successful!
OpenCV version: 4.x.x
YOLOv8: Ready ✓

Estimated time: 2-3 minutes
```

### 📍 CELL 2: Load and Explore Dataset
```
What it does:
- Finds your dataset in Google Drive
- Shows sample images
- Displays dataset statistics

How to run:
1. Click on the cell
2. Click the ▶️ (Play) button
3. Wait for output and images to display

Expected output:
📊 DATASET STATISTICS
Total images found: XXXX
Disease Classes:
  • Disease1: XXX images
  • Disease2: XXX images
  • Disease3: XXX images
  • Disease4: XXX images

And 6 sample leaf images will display

Estimated time: 1-2 minutes

⚠️ If "No images found":
- Modify DATASET_PATH variable
- Use an alternative path format
- Check your Google Drive folder permissions
```

### 📍 CELL 3: Data Augmentation Pipeline  
```
What it does:
- Creates 5 different augmentation strategies
- Generates 6000-8000 images from original dataset
- Saves augmented images to Google Drive

How to run:
1. Click on the cell
2. Click the ▶️ (Play) button
3. WAIT - This takes 30-45 minutes!
4. You'll see progress: "✓ Processed X/Y images"
5. When complete: "✅ AUGMENTATION COMPLETE!"

Expected output:
🎨 DATA AUGMENTATION PIPELINE
Total augmented images: 6000-8000

Estimated time: 30-45 minutes ⏳

⚠️ If cell seems stuck:
- Don't interrupt it
- Check if the progress indicator is still updating
- GPU utilization should be visible in sidebar
```

### 📍 CELL 4: Prepare Data for YOLO
```
What it does:
- Organizes augmented images in YOLO format
- Creates train/val/test splits (80/10/10)
- Generates dataset.yaml configuration

How to run:
1. Click on the cell
2. Click the ▶️ (Play) button
3. Wait for "✅ YOLO dataset ready" message

Expected output:
📦 PREPARING YOLO DATASET
Total augmented images: XXXX

SPLIT STATISTICS:
TRAIN: XXXX images
VAL: XXX images
TEST: XXX images

✅ YOLO dataset ready at: /content/yolo_dataset

Estimated time: 5 minutes
```

### 📍 CELL 5: Train YOLO Model
```
What it does:
- Loads YOLOv8 pretrained model
- Trains on your augmented dataset
- Saves best model checkpoints

How to run:
1. Click on the cell
2. Click the ▶️ (Play) button
3. WAIT - This takes 45-90 minutes!
4. Watch for training progress (epoch 1/50, 2/50, etc.)
5. When complete: "✅ Training completed successfully!"

Expected output:
🚀 TRAINING YOLO MODEL
Model: YOLOv8n
Epochs: 50
Batch Size: 16

Epoch 1/50: 100%|████| 20/20 [00:15<00:00, 1.30it/s]
[output showing loss values...]
Epoch 2/50: 100%|████| 20/20 [00:14<00:00, 1.35it/s]
[... continues for 50 epochs ...]

✅ Training completed successfully!

Estimated time: 45-90 minutes ⏳

⚠️ TROUBLESHOOTING:
- "Out of memory": Reduce BATCH_SIZE from 16 to 8
- "CUDA error": Restart runtime and re-run
- Very slow: Check GPU is enabled (Runtime menu)
```

### 📍 CELL 6: Run Disease Detection (Inference)
```
What it does:
- Loads trained model
- Runs predictions on test images
- Displays results with disease predictions

How to run:
1. Click on the cell
2. Click the ▶️ (Play) button
3. Wait for predictions to display

Expected output:
🔍 DISEASE DETECTION - INFERENCE
Running inference on XX test images...

6 sample images will display with:
- Predicted disease name
- Confidence percentage
- Actual disease name
- Green border if correct, Red if wrong

Estimated time: 5-10 minutes
```

### 📍 CELL 7: Evaluate Model Results
```
What it does:
- Calculates accuracy, precision, recall
- Creates confusion matrix
- Generates detailed performance report

How to run:
1. Click on the cell
2. Click the ▶️ (Play) button
3. Wait for evaluation metrics

Expected output:
📊 MODEL EVALUATION
Evaluating on XX test images...

Overall Accuracy: XX.XX%

Classification Report:
              precision    recall  f1-score   support
    Disease1       0.92      0.88      0.90       100
    Disease2       0.85      0.90      0.87        95
    Disease3       0.88      0.92      0.90        98
    Disease4       0.90      0.85      0.87       102

Confusion matrix will display as heatmap

Estimated time: 3-5 minutes
```

### 📍 CELL 8: Final Summary & Download Results
```
What it does:
- Shows complete pipeline summary
- Lists all saved files with locations
- Explains how to use the trained model

How to run:
1. Click on the cell
2. Click the ▶️ (Play) button
3. Read the complete summary

Expected output:
🎉 YOLO DISEASE DETECTION PIPELINE COMPLETE! 🎉

DATASET TRANSFORMATION:
  • Original images: XXX per class
  • Augmented images: 6000-8000 total
  • Training images: XXXX
  • Validation images: XXX
  • Test images: XXX

MODEL PERFORMANCE:
  • Overall Accuracy: XX.XX%
  • Model: YOLOv8n
  • Training epochs: 50

OUTPUT FILES SAVED:
  ✓ best_model.pt (trained model)
  ✓ confusion_matrix.png
  ✓ evaluation_report.txt
  ✓ inference_results.png

HOW TO USE THE MODEL:
[Code example shown]

Estimated time: 1 minute
```

---

## 📊 TOTAL RUNTIME ESTIMATE

| Phase | Time | Can Skip? |
|-------|------|----------|
| CELL 1: Setup | 2-3 min | ❌ Required |
| CELL 2: Explore | 1-2 min | ✅ Optional |
| CELL 3: Augmentation | 30-45 min | ❌ Critical |
| CELL 4: YOLO Format | 5 min | ❌ Required |
| CELL 5: Training | 45-90 min | ❌ Critical |
| CELL 6: Inference | 5-10 min | ✅ Optional |
| CELL 7: Evaluation | 3-5 min | ✅ Optional |
| CELL 8: Summary | 1 min | ✅ Optional |
| **TOTAL** | **2-3 hours** | - |

---

## ⚠️ IMPORTANT WARNINGS

### GPU Is Essential
- **WITH GPU:** 2-3 hours total
- **WITHOUT GPU:** 8-12 hours total
- **Always enable GPU** (Runtime → Change runtime type)

### Google Drive Space
- Original dataset: ~1-2 GB
- Augmented dataset: ~4-5 GB  
- Results: ~1 GB
- **Total needed: ~10 GB**

### Connection Loss
- If Colab disconnects: Click "Reconnect" (top right)
- Work is NOT lost - code continues running
- Check Google Drive to see progress

### Long Running Cells
- Don't close the browser tab during Cell 3 or 5
- Don't press "Interrupt" unless absolutely necessary
- Let it run to completion

---

## 🎯 AFTER RUNNING ALL CELLS

### Download Results from Google Drive
1. Open Google Drive: https://drive.google.com
2. Navigate to: **YOLO_Results** folder
3. Download these files:
   - **best_model.pt** - Your trained model
   - **confusion_matrix.png** - Performance chart
   - **evaluation_report.txt** - Detailed metrics

### Use Model for New Predictions
See the code example in CELL 8 to run predictions on new leaf images!

---

## 🆘 TROUBLESHOOTING

| Problem | Solution |
|---------|----------|
| Google Drive not mounting | Run Cell 1 again, grant permissions |
| "No images found" error | Update DATASET_PATH in Cell 2 |
| Out of memory error | Reduce BATCH_SIZE (Cell 5) from 16 to 8 or 4 |
| Colab connection lost | Click "Reconnect" button, work continues |
| Very slow execution | Make sure GPU is enabled in Runtime settings |
| CUDA/GPU errors | Restart runtime (Runtime → Restart runtime) |
| Module not found | Re-run Cell 1 (dependencies installation) |

---

## ✅ NEXT STEPS

1. ✔️ Run all cells in order
2. ✔️ Download best_model.pt from Google Drive
3. ✔️ Use trained model on your own banana leaf images
4. ✔️ For better accuracy: increase EPOCHS from 50 to 100
5. ✔️ Consider using larger model (YOLOv8s, YOLOv8m)